# Metrological Analysis of Historical Scripts
### Companion notebook to *Leveraging Morphology for Historical Script Metrological Analysis*

To reproduce the paper figures, download and extract the experimental dataset from [Zenodo](https://doi.org/10.5281/zenodo.15282371) into the base folder. The figure below summarises the dataset structure and the components used throughout the pipeline.

The dataset consists of cropped text-line images and an `annotation.json` file containing transcription labels and metadata fields describing the document context. The granularity and nature of these metadata depend on the research question: in this study they include folio, zone type, line type, and graphic profile. The unit of analysis is the page.

The `input/` directory contains the raw outputs of the DTLR for paleography architecture:
- one folder of extracted prototypes per page and per character
- a `character_measurements/` folder with one `.json` per text line, containing character-level bounding-box measurements (see Figure below)

![diagram](./media/explanation_dataset_components.png)

This notebook has two purposes, which are interleaved throughout:
- **Reproducing the paper** — each section references the corresponding paper section or figure explicitly.
- **Applying the pipeline to your own data** — wherever the paper-specific settings appear, we explain what to change. A short guide is provided at the top of each major section.

## Adapting this pipeline to your own dataset

The pipeline was designed to generalise beyond the BnF fr. 2813 case study. This notebook shows how. The key parameters are all gathered in a single `StudyConfig` object (Step 1 below); changing them is sufficient for most datasets.

**What you need at minimum:**
- An `annotation.json` mapping each line image filename to its metadata (folio, script, line type, zone — depends on your analysis)
- A `character_measurements/` folder — one subfolder per document, one `.json` per transcribed line (standard output from DTLR)
- Optionally, `text_line_images/` if you want to crop and visualise individual character bboxes
- Optionally, `prototypes/` if you want the prototype overlay on crossed graphs (standard output from DTLR)

**The main knobs:**

| Parameter | What it controls | Paper value |
|---|---|---|
| `group_field` | Annotation field used to colour-group documents | `"gp"` (Graphic Profile) |
| `line_type_filter` | Which line type to include | `"DefaultLine"` |
| `line_selection_mode` | Zone/column selection | `"recto_verso"` (outer columns only) |
| `outlier_std_threshold` | Bbox outlier filter (§3.2 *Discarding errors*) | `4.0` |
| `group_order` | Left-to-right manuscript order on x-axis | `["GP1","GP2","GP3","GP4"]` |

Everything else — metrics, plots, filters — follows from these.

## Imports

In [ ]:
import os
import re
import json
import glob
import pandas as pd
import numpy as np
from tqdm import tqdm
from pathlib import Path
from tqdm.notebook import tqdm
from collections import defaultdict

from PIL import Image
from skimage.io import imread
from skimage.color import rgb2gray
from skimage.transform import resize

import matplotlib.pyplot as plt
from matplotlib import colors, cm

from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D

import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import matplotlib
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.family'] = 'STIXGeneral'

plt.rcParams['text.usetex'] = True

import sys
sys.path.append(str(Path("scripts")))

from metrics_viz import (StudyConfig, FolioIndex, m, compute_letter_metrics, compute_bigram_metrics, 
                                 compute_word_metrics, plot_linear, plot_cross)
from build_corpus import (build_doc_mappings, build_line_mappings, build_character_index_map, 
                        build_and_crop_working_corpus, compute_working_corpus_statistics)

## File paths

Paths are resolved automatically from the dataset structure. To use your own data, set `base_dir` to your dataset root and adjust the subpath variables below to match your folder layout.

In [ ]:
from pathlib import Path

# start from current working directory
current = Path().resolve()

# walk upward until we find a marker of the repo root
for parent in [current] + list(current.parents):
    if (parent / "input").exists() and (parent / "dataset").exists():
        base_dir = parent
        break
else:
    raise FileNotFoundError("Could not locate project root (input/ and dataset/ missing)")

In [ ]:
# --- Subpaths ---
#inputs
prototypes                  = base_dir / "input/prototypes"
character_mapping           = base_dir / "input/transcribe.json"
character_measurements      = base_dir / "input/character_measurements/"

#dataset
annotation_json             = base_dir / "dataset/annotation.json"
text_line_images            = base_dir / "dataset/images"

#save results
fig_output_dir              = base_dir / "results/figures"
cropped_character_bboxes    = base_dir / "results/cropped_character_bboxes"

## (Optional) Step 0 — Refining bounding boxes from prototype masks
*Paper §3.2 — "Refined boxes for improved consistency with human perception"*

The predicted bounding boxes may include near-white border pixels corresponding to parts of the character that vary between instances. Following the strategy described in §3.2, we compute a tighter bounding box from the prototype mask and transport it to each character instance.

**run**, `correct_bboxes_pipeline()` once on your prototype folder and your `character_measurements/` folder. It writes a corrected `character_measurements_corrected/` folder; point `character_measurements` at it in the paths cell above, and the rest of the pipeline is unchanged.

**Skip this cell entirely if you do not want to produce fitted bboxes.**

In [ ]:
from fit_bboxes import fit_bboxes_pipeline, visualise_fitted_bboxes, plot_before_after

# ── Run fitting  ─────────────────────────────────────────────
dict_ratios = fit_bboxes_pipeline(
    proto_folder          = base_dir / "input/prototypes/M0_without_aspect_ratio",
    input_measurements    = base_dir / "input/character_measurements",
    output_measurements   = base_dir / "input/character_measurements_fitted",
    output_cropped_protos = None,   # set to base_dir / "output/prototype_cropped" to visualise the masks
    threshold             = 0.85 * 255, #decide on the mask threshold.
    sprite_range          = range(0, 120),
    save_ratios_csv       = None, #set to base_dir / "output/prototype_bbox_proportions.csv" to track proportions
)

# ── Point the pipeline at the fitted bbox folder instead of original ───────────────────────────────
character_measurements = base_dir / "input/character_measurements_fitted"

# ── (Optional) Visualise fitted bboxes on line images ─────────────────────
# visualise_fitted_bboxes(
#     fitted_measurements = base_dir / "input/character_measurements_fitted",
#     line_images         = base_dir / "dataset/images",
#     output_vis_folder   = base_dir / "output/line_level_with_boxes",
# )

# ── (Optional) Side-by-side before/after for a single line ────────────────────────
# plot_before_after(
#     doc                 = "btv1b84472995_f009",
#     line                = "btv1b84472995_f009_eSc_line_00acb699",
#     orig_measurements   = base_dir / "input/character_measurements",
#     fitted_measurements = base_dir / "input/character_measurements_fitted",
#     line_images         = base_dir / "dataset/images",
# )

## Step 1 — Configuration
*Paper §3.3 — Measures and visualisations; §4.1 — Case study and dataset*

All pipeline parameters are defined once in `StudyConfig` and passed to every function. The values below reproduce the paper. Adapt the *Generalisation* block for your own dataset — everything else (metrics, plots, filters) follows automatically.

### Corpus filters and visualisation settings

`StudyConfig` controls three things at once:
1. **Which lines enter the corpus** — via `line_type_filter`, `line_selection_mode`, and `outlier_std_threshold` (§3.2 *Discarding errors*: bboxes deviating more than 4σ from the per-character mean are excluded)
2. **How documents are grouped and ordered** — via `group_field` and `group_order` (the four Graphic Profiles GP1–GP4 in the paper, §4.1)
3. **How plots look** — markers per character/bigram, colours per group

`FolioIndex` is built from the config and resolves all folio-level mappings (natural sort order, group colours, x-axis positions). It is used by every plot function.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# Define ONE config — used by build_and_crop_working_corpus, FolioIndex,
# the metric functions, and the plot functions. No duplication anywhere.
#
# Defaults below reproduce the paper analysis. Adjust the "Generalisation"
# block to adapt to other datasets.
# ─────────────────────────────────────────────────────────────────────────────
config = StudyConfig(
    annotation_json        = annotation_json,
    character_measurements = character_measurements,
    fig_output_dir         = fig_output_dir,
    prototypes             = prototypes,

    # ── Generalisation: change these to adapt to your dataset ────────────────
    group_field            = "gp",           # None | "script" | "scribe" | ...
    doc_key_parts          =  2,               # parts of the filename forming the doc key
    line_type_filter       = "DefaultLine",   # None to keep all lines
    line_selection_mode    = "recto_verso",   # None | "MainZone#1" | "MainZone#2" | "recto_verso"
    filter_border_bboxes   = True,

    # ── BBox outlier filter ──────────────────────────────────────────────────
    bbox_filter_mode       = "threshold",
    outlier_std_threshold  = 4,

    # ── Gathering ranges shown in every linear plot — change only if needed ──
    highlight_ranges = [
        ('1r',   '7v'),
        ('170r', '178r'), ('178r', '185v'),
        ('401r', '409r'), ('409r', '416v'),
        ('467r', '474v'), ('474v', '480v'),
    ],
    gathering_labels = {
        ('1r',   '7v'):   "Gath. I",
        ('170r', '178r'): "Gath. XXII",
        ('178r', '185v'): "Gath. XXIII",
        ('401r', '409r'): "Gath. LII",
        ('409r', '416v'): "Gath. LIII",
        ('467r', '474v'): "Gath. LXII",
        ('474v', '480v'): "Gath. LXIII",
    },

    # ── Markers (sensible defaults; override only if needed) ─────────────────
    letter_markers = {'a': 's', 't': 'x', 'd': '*', 'e': 'P', 'n': 'o'},
    bigram_markers = {'en': 'o', 'de': 'p', 'le': '^'},
)

idx = FolioIndex(config)
print(f"Corpus has {len(idx.all_folios_sorted)} total folios/pages")
print(f"Resolved group colours: {idx.group_colors}")


## Step 2 — Dataset metadata mappings

We build three mappings from `annotation.json`:
- **character → sprite index** — links each character in the alphabet to its prototype image
- **document → group / folio** — used for colour-coding and x-axis ordering
- **line → type / zone** — used for corpus filtering

For your own data: if some of these fields are absent from your annotation, set the corresponding config parameter to `None` — the pipeline degrades gracefully.

In [ ]:
# ── Mappings (built once, reused everywhere) ──────────────────────────────
character_sprite_map = build_character_index_map(character_mapping)
doc_mappings,  _     = build_doc_mappings(annotation_json)
line_mappings, _     = build_line_mappings(annotation_json)

## Step 3 — Build the working corpus
*Paper §3.2 — "Discarding errors"; §4.1 — dataset statistics*

`build_and_crop_working_corpus` reads all DTLR prediction JSONs and applies two filters in sequence:

1. **Nomatch filter** — discards any bbox neighbouring a transcription error (insertion, deletion, replacement detected by dynamic-programming alignment against the ground truth, §3.2)
2. **Outlier bbox filter** — discards, per document and per character, any bbox whose width or height deviates by more than `outlier_std_threshold` σ from the document mean (§3.2 — "*This only excludes a very small proportion of the data, 0.1%*")

The result is `corpus` (in-memory, used for letter metrics), `stats` (per-group counts), and `metadata_rows` (one row per bbox, used to propagate filter decisions into bigram and word metrics).

Set `crop_images=True` to also save cropped bbox PNGs to disk — useful for qualitative inspection but not needed for metrics.

In [ ]:
corpus, stats, metadata_rows = build_and_crop_working_corpus(
    character_measurements   = character_measurements,
    text_line_images         = text_line_images,
    cropped_character_bboxes = cropped_character_bboxes,
    annotation_json          = annotation_json,
    config                   = config,                # single source of truth
    target_characters        = None,                  # None = all characters
    character_sprite_map     = character_sprite_map,
    crop_images              = None,    # True saves a folder with kept/discarded bbox PNGs
    #crop_characters          = ["t", "a", "e", "d", "n", "c", "u"]    #if the above is True, you can precise characters
)

df_gp, df_char = compute_working_corpus_statistics(
    corpus, stats, group_label="script"      # change to "Script"/"Document"/... for other groupings
)


## Step 4 — Compute metrics
*Paper §3.3 — "Measures": width, aspect ratio, bigram distance, word separation*

We accumulate all metrics into a single wide DataFrame `df` (one row per folio). Three types of measures are computed, corresponding to Figure 1 and §3.3:

![diagram](./media/measures_definition.jpeg)

- **Letter aspect ratio** `w/h` — mean and CV per character per folio (Figure 6a, Figure 7a–b)
- **Bigram edge-to-edge distance** `d_b` — signed distance between adjacent character bboxes, normalised by `w_m/2` (half the mean width of lowercase *m*, §3.3; Figure 6b, Figure 7c–d)
- **Word separation** `d_w` — distance between the last bbox of one word and the first of the next, normalised by `w_m/2` (Figure 6c)

For bigram and word metrics, only bboxes that survived both filters (nomatch + outlier) contribute to the measurement — ensured by passing `metadata_rows` to `compute_bigram_metrics` and `compute_word_metrics`.

**For your own data**: replace the character and bigram lists with your top-N most frequent ones (use the top-N helper cell below if needed).




In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
#  Compute metrics (Accumulate into one wide DataFrame)
# ─────────────────────────────────────────────────────────────────────────────
df = idx.base_df()

# letters
for char in ["n", "a", "t", "d", "e"]:
    df = compute_letter_metrics(df, corpus, char, config)

# bigrams (case-sensitive: "de" matches only lowercase d+e, "De" only D+e, etc.)
for bigram in ["en", "de", "le"]:
    df = compute_bigram_metrics(df, bigram, config, idx, metadata_rows=metadata_rows)

# word
df = compute_word_metrics(df, config, idx, metadata_rows=metadata_rows)

# Sanity check using the m() helper for readability
ar_n = m("letter", "n", "AR")           # -> "mean_ar_n"
missing = df[df[ar_n].isna()]["folio"].tolist()
print(f"Pages with no data for {ar_n}: {missing}")
print(f"{df[ar_n].notna().sum()} units in the working corpus")
print(df.shape)
df.head()

## Step 5 — Visualisations
*Paper §3.3 — "Visualisation types"; Figures 6 and 7*

We use two plot types described in §3.3 and illustrated in Figure 3:

![diagram](./media/dummy_graphs.jpeg)


- **Linear graphs** (`plot_linear`) — evolution of one or more measures across ordered folios. Point colour = graphic profile; point opacity = number of occurrences (robustness indicator).
- **Crossed graphs** (`plot_cross`) — two measures against each other, one point per folio. Enables correlation analysis and finer group separation.

In both, the x-axis follows the folio order defined by `group_order` in the config (GP1 → GP2 → GP3 → GP4 in the paper). Highlighted bands show gatherings; dotted lines mark gathering boundaries.

### Linear graphs — reproducing Figures 6a, 6b, 6c from paper §4.3

- **Figure 6a** — letter "a", "n", "t", "d", "e" aspect ratios.

In [ ]:
# ── LINEAR PLOTS ─────────────────────────────────────────────────────────────
# Letter AR: a, n, t, d, e
plot_linear(
    df,
    y_metric=[m("letter", c, "AR") for c in ["a", "n", "t", "d", "e"]],
    config=config, idx=idx,
    y_label_override="Character Aspect Ratio",
)

- **Figure 6b** — bigram "en", "de", "le" distances

In [ ]:
# Bigram edge-to-edge distance: en, de, le
plot_linear(
    df,
    y_metric=[m("bigram", b, "normalized distance") for b in ["en", "de", "le"]],
    config=config, idx=idx,
    y_label_override="Bigram Distance",
)

- **Figure 6c** — word separation

In [ ]:
# Word separation
plot_linear(
    df,
    y_metric=m("word", "normalized distance"),
    config=config, idx=idx,
)

### Crossed graphs — reproducing Figures 7a, 7b, 7c, 7d

The four plots below reproduce Figure 7 from §4.3:
- **Figure 7a** — µ[AR] of ⟨a⟩ × µ[AR] of ⟨n⟩

In [ ]:
# AR of n vs AR of a
plot_cross(
    df,
    x_metric=m("letter", "n", "AR"),
    y_metric=m("letter", "a", "AR"),
    config=config, idx=idx,
)


- **Figure 7b** — µ[AR] of ⟨t⟩ × CV[AR] of ⟨t⟩

In [ ]:
# AR of t vs CV of t — with prototype overlay
plot_cross(
    df,
    x_metric=m("letter", "t", "AR"),
    y_metric=m("letter", "t", "AR CV"),
    config=config, idx=idx,
    overlay_prototypes=True, proto_char="t",
    character_sprite_map=character_sprite_map,
)


- **Figure 7c** — µ[d_b] × µ[AR] of ⟨en⟩: no clear proportionality between spacing and letter morphology

In [ ]:
plot_cross(
    df,
    x_metric=m("bigram", "en", "normalized distance"),
    y_metric=m("bigram", "en", "AR"),
    config=config, idx=idx,
)

- **Figure 7d** — µ[d_b] × CV[d_b] of ⟨de⟩

In [ ]:
# Edge-to-edge distance of "de" vs CV of distance of "de"
plot_cross(
    df,#[df["folder"] != "btv1b84472995_f015"],
    x_metric=m("bigram", "de", "normalized distance"),
    y_metric=m("bigram", "de", "distance CV"),
    config=config, idx=idx,
)